In [1]:
# Sel ini menghasilkan TIGA dataset cabang (kota) terpisah untuk Tugas Mandiri Pertemuan 3
import numpy as np
import pandas as pd

kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-08-01", "2026-08-31", freq="D")

cabang_kota = {"Magelang": 101, "Yogyakarta": 202, "Semarang": 303}

for kota, seed in cabang_kota.items():
    np.random.seed(seed)  # seed berbeda tiap kota agar datanya bervariasi, namun tetap konsisten/reproducible
    n = 200
    data_cabang = {
        "order_id": [f"{kota[:3].upper()}-{2000 + i}" for i in range(n)],
        "tanggal": np.random.choice(tanggal_range, size=n),
        "kategori": np.random.choice(kategori_list, size=n, p=[0.25, 0.25, 0.20, 0.15, 0.15]),
        "unit_terjual": np.random.randint(1, 8, size=n),
        "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000, 250000], size=n),
        "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    }
    df_cabang = pd.DataFrame(data_cabang)
    df_cabang["kota"] = kota
    nama_file = f"transaksi_{kota.lower()}.csv"
    df_cabang.to_csv(nama_file, index=False)
    print(f"Berkas '{nama_file}' berhasil dibuat: {df_cabang.shape[0]} baris")

print("\nKetiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.")

Berkas 'transaksi_magelang.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_yogyakarta.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_semarang.csv' berhasil dibuat: 200 baris

Ketiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.


In [2]:
#A
# Membuat direktori raw dan processed di HDFS
!hdfs dfs -mkdir -p /user/yue/ecommerce/raw
!hdfs dfs -mkdir -p /user/yue/ecommerce/processed

# Menampilkan hasil untuk membuktikan struktur terbentuk
!hdfs dfs -ls -R /user/yue/ecommerce

drwxr-xr-x   - yue supergroup          0 2026-09-09 23:29 /user/yue/ecommerce/processed
drwxr-xr-x   - yue supergroup          0 2026-09-09 23:29 /user/yue/ecommerce/raw
-rw-r--r--   1 yue supergroup      12329 2026-09-09 23:29 /user/yue/ecommerce/raw/transaksi_magelang.csv
-rw-r--r--   1 yue supergroup      12154 2026-09-09 23:29 /user/yue/ecommerce/raw/transaksi_semarang.csv
-rw-r--r--   1 yue supergroup      12684 2026-09-09 23:29 /user/yue/ecommerce/raw/transaksi_yogyakarta.csv


In [3]:
#B                         
# Mengunggah ketiga berkas CSV ke direktori raw
!hdfs dfs -put transaksi_magelang.csv transaksi_yogyakarta.csv transaksi_semarang.csv /user/yue/ecommerce/raw/

# Menampilkan isi direktori beserta ukurannya
!hdfs dfs -ls /user/yue/ecommerce/raw/

put: `/user/yue/ecommerce/raw/transaksi_magelang.csv': File exists
put: `/user/yue/ecommerce/raw/transaksi_yogyakarta.csv': File exists
put: `/user/yue/ecommerce/raw/transaksi_semarang.csv': File exists
Found 3 items
-rw-r--r--   1 yue supergroup      12329 2026-09-09 23:29 /user/yue/ecommerce/raw/transaksi_magelang.csv
-rw-r--r--   1 yue supergroup      12154 2026-09-09 23:29 /user/yue/ecommerce/raw/transaksi_semarang.csv
-rw-r--r--   1 yue supergroup      12684 2026-09-09 23:29 /user/yue/ecommerce/raw/transaksi_yogyakarta.csv


In [10]:
#C
import os
import subprocess
import pandas as pd
from IPython.display import display

# Set classpath & environment Java/Hadoop
hadoop_classpath = subprocess.run(["hadoop", "classpath", "--glob"], capture_output=True, text=True).stdout.strip()
os.environ["CLASSPATH"] = hadoop_classpath
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

# Path HDFS disesuaikan dengan username kamu (yue)
path_hdfs = "hdfs://localhost:9000/user/yue/ecommerce/raw/"

# Membaca data dari HDFS
df_mag = pd.read_csv(f"{path_hdfs}transaksi_magelang.csv")
df_yog = pd.read_csv(f"{path_hdfs}transaksi_yogyakarta.csv")
df_sem = pd.read_csv(f"{path_hdfs}transaksi_semarang.csv")

# Menggabungkan data
df_gabungan = pd.concat([df_mag, df_yog, df_sem], ignore_index=True)

# Menampilkan tabel secara rapi
print("=== TABEL MAGELANG ===")
display(df_mag.head())

print("\n=== TABEL YOGYAKARTA ===")
display(df_yog.head())

print("\n=== TABEL SEMARANG ===")
display(df_sem.head())

print("\n=== BUKTI PENGGABUNGAN (Jumlah Transaksi per Kota) ===")
print(df_gabungan['kota'].value_counts())

=== TABEL MAGELANG ===


,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota
0,MAG-2000,2026-08-12,Fashion,1,50000,Transfer Bank,Magelang
1,MAG-2001,2026-08-18,Elektronik,7,25000,COD,Magelang
2,MAG-2002,2026-08-07,Elektronik,7,25000,E-Wallet,Magelang
3,MAG-2003,2026-08-24,Rumah Tangga,6,25000,COD,Magelang
4,MAG-2004,2026-08-30,Fashion,7,100000,Transfer Bank,Magelang



=== TABEL YOGYAKARTA ===


,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota
0,YOG-2000,2026-08-20,Fashion,5,250000,E-Wallet,Yogyakarta
1,YOG-2001,2026-08-24,Fashion,4,150000,Kartu Kredit,Yogyakarta
2,YOG-2002,2026-08-29,Rumah Tangga,4,100000,E-Wallet,Yogyakarta
3,YOG-2003,2026-08-13,Fashion,7,250000,E-Wallet,Yogyakarta
4,YOG-2004,2026-08-09,Elektronik,7,75000,COD,Yogyakarta



=== TABEL SEMARANG ===


,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota
0,SEM-2000,2026-08-28,Elektronik,3,100000,E-Wallet,Semarang
1,SEM-2001,2026-08-29,Kesehatan & Kecantikan,4,25000,Transfer Bank,Semarang
2,SEM-2002,2026-08-19,Rumah Tangga,1,25000,COD,Semarang
3,SEM-2003,2026-08-02,Makanan & Minuman,2,75000,E-Wallet,Semarang
4,SEM-2004,2026-08-05,Elektronik,6,25000,Transfer Bank,Semarang



=== BUKTI PENGGABUNGAN (Jumlah Transaksi per Kota) ===
kota
Magelang      200
Yogyakarta    200
Semarang      200
Name: count, dtype: int64


In [11]:
#D

# 1. Tambah kolom total_pendapatan
df_gabungan['total_pendapatan'] = df_gabungan['unit_terjual'] * df_gabungan['harga_satuan']

# 2. Buat tabel ringkasan
df_ringkasan = df_gabungan.groupby(['kota', 'kategori'])['total_pendapatan'].sum().reset_index()

# Menampilkan tabel hasil olahan secara rapi
print("=== TABEL DATA GABUNGAN BERSIH ===")
display(df_gabungan.head())

print("\n=== TABEL RINGKASAN KOTA & KATEGORI ===")
display(df_ringkasan)

# 3. Simpan CSV lokal & unggah ke HDFS processed
df_gabungan.to_csv("data_gabungan_bersih.csv", index=False)
df_ringkasan.to_csv("ringkasan_kota_kategori.csv", index=False)

!hdfs dfs -put -f data_gabungan_bersih.csv ringkasan_kota_kategori.csv /user/yue/ecommerce/processed/
!hdfs dfs -ls /user/yue/ecommerce/processed/

=== TABEL DATA GABUNGAN BERSIH ===


,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota,total_pendapatan
0,MAG-2000,2026-08-12,Fashion,1,50000,Transfer Bank,Magelang,50000
1,MAG-2001,2026-08-18,Elektronik,7,25000,COD,Magelang,175000
2,MAG-2002,2026-08-07,Elektronik,7,25000,E-Wallet,Magelang,175000
3,MAG-2003,2026-08-24,Rumah Tangga,6,25000,COD,Magelang,150000
4,MAG-2004,2026-08-30,Fashion,7,100000,Transfer Bank,Magelang,700000



=== TABEL RINGKASAN KOTA & KATEGORI ===


,kota,kategori,total_pendapatan
0,Magelang,Elektronik,18775000
1,Magelang,Fashion,27750000
2,Magelang,Kesehatan & Kecantikan,17375000
3,Magelang,Makanan & Minuman,17525000
4,Magelang,Rumah Tangga,12200000
5,Semarang,Elektronik,21425000
6,Semarang,Fashion,26425000
7,Semarang,Kesehatan & Kecantikan,13525000
8,Semarang,Makanan & Minuman,19750000
9,Semarang,Rumah Tangga,10975000


Found 2 items
-rw-r--r--   1 yue supergroup      41257 2026-09-10 00:29 /user/yue/ecommerce/processed/data_gabungan_bersih.csv
-rw-r--r--   1 yue supergroup        530 2026-09-10 00:29 /user/yue/ecommerce/processed/ringkasan_kota_kategori.csv
